<a href="https://colab.research.google.com/github/PromyotKatarat/AI_agent_engineering/blob/main/The_Tool_Interface_%E2%80%94_Why_Agents_Need_Structured_I_O.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# 1. ดึงโปรเจกต์มาทั้งหมดก่อน
!git clone https://github.com/rohitg00/ai-engineering-from-scratch.git

Cloning into 'ai-engineering-from-scratch'...
remote: Enumerating objects: 15947, done.
remote: Counting objects: 100% (6813/6813), done.
remote: Compressing objects: 100% (2490/2490), done.
remote: Total 15947 (delta 4710), reused 4326 (delta 4323), pack-reused 9134 (from 2)
Receiving objects: 100% (15947/15947), 8.71 MiB | 13.12 MiB/s, done.
Resolving deltas: 100% (7331/7331), done.


In [3]:
# 2. ย้ายตำแหน่งเข้าไปที่โฟลเดอร์ของบทเรียนนี้โดยตรง
%cd ai-engineering-from-scratch/phases/13-tools-and-protocols/01-the-tool-interface/code/

# 3. เช็คดูว่ามีไฟล์โครงสร้างหลักอยู่ครบไหม (ต้องมี main.py, verify.py)
!ls -la

/content/ai-engineering-from-scratch/phases/13-tools-and-protocols/01-the-tool-interface/code
total 28
drwxr-xr-x 2 root root 4096 Jun  6 12:32 .
drwxr-xr-x 7 root root 4096 Jun  6 12:32 ..
-rw-r--r-- 1 root root 7844 Jun  6 12:32 main.py
-rw-r--r-- 1 root root 8257 Jun  6 12:32 main.ts


In [4]:
"""Phase 13 Lesson 01 - the tool interface, four-step loop, no LLM.

Implements the describe -> decide -> execute -> observe cycle used by every
2026 tool-calling stack (OpenAI, Anthropic, Gemini, MCP, A2A). The "decide"
step is faked with a keyword router so the loop runs offline; replace it with
any real provider in Lesson 02.

The harness:
  - registers three tools (add, get_time, get_weather)
  - validates tool-call arguments against a minimal JSON Schema subset
  - prints each step so you can read the choreography
  - bounds iteration at MAX_TURNS to prevent runaway loops

Run: python code/main.py
"""

from __future__ import annotations

import datetime as dt
import json
import re
import time
import uuid
from dataclasses import dataclass
from typing import Any, Callable


MAX_TURNS = 5


@dataclass
class Tool:
    name: str
    description: str
    input_schema: dict
    executor: Callable[[dict], Any]
    consequential: bool = False


def tool_add(args: dict) -> dict:
    return {"sum": args["a"] + args["b"]}


def tool_get_time(args: dict) -> dict:
    tz = args.get("timezone", "UTC")
    now = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
    return {"now": now, "timezone": tz}


def tool_get_weather(args: dict) -> dict:
    fake = {"Bengaluru": 28, "Tokyo": 12, "Zurich": 4, "Lagos": 31}
    city = args["city"]
    units = args.get("units", "celsius")
    temp = fake.get(city, 20)
    return {"city": city, "temp": temp, "units": units}


REGISTRY: list[Tool] = [
    Tool(
        name="add",
        description=(
            "Use when the user asks for the sum of two numbers. "
            "Do not use for subtraction, product, or symbolic algebra."
        ),
        input_schema={
            "type": "object",
            "properties": {
                "a": {"type": "number"},
                "b": {"type": "number"},
            },
            "required": ["a", "b"],
        },
        executor=tool_add,
    ),
    Tool(
        name="get_time",
        description=(
            "Use when the user asks what time it is. "
            "Do not use for historical dates or future scheduling."
        ),
        input_schema={
            "type": "object",
            "properties": {
                "timezone": {"type": "string"},
            },
            "required": [],
        },
        executor=tool_get_time,
    ),
    Tool(
        name="get_weather",
        description=(
            "Use when the user asks about current conditions in a named city. "
            "Do not use for forecasts or historical weather data."
        ),
        input_schema={
            "type": "object",
            "properties": {
                "city": {"type": "string"},
                "units": {"type": "string", "enum": ["celsius", "fahrenheit"]},
            },
            "required": ["city"],
        },
        executor=tool_get_weather,
    ),
]


def validate(schema: dict, value: Any) -> list[str]:
    errors: list[str] = []
    t = schema.get("type")
    if t == "object":
        if not isinstance(value, dict):
            return [f"expected object, got {type(value).__name__}"]
        for field in schema.get("required", []):
            if field not in value:
                errors.append(f"missing required field '{field}'")
        for key, sub in schema.get("properties", {}).items():
            if key in value:
                errors.extend(validate(sub, value[key]))
        return errors
    if t == "number" and not isinstance(value, (int, float)):
        errors.append(f"expected number, got {type(value).__name__}")
    if t == "string" and not isinstance(value, str):
        errors.append(f"expected string, got {type(value).__name__}")
    if "enum" in schema and value not in schema["enum"]:
        errors.append(f"value {value!r} not in enum {schema['enum']}")
    return errors


def fake_decide(user_msg: str, history: list[dict]) -> dict:
    """Stand-in for the model. Routes by keyword so the loop runs offline.

    Production substitute: swap this for provider.chat.completions.create with
    tools=[t.input_schema for t in REGISTRY]. Same return shape.
    """
    last = history[-1] if history else {}
    if last.get("role") == "tool":
        return {"content": f"Final answer built from tool output: {last.get('content')}"}
    msg = user_msg.lower()
    if re.search(r"\b(add|sum|plus)\b", msg):
        nums = [float(n) for n in re.findall(r"-?\d+\.?\d*", msg)]
        if len(nums) >= 2:
            return {
                "tool_calls": [
                    {
                        "id": f"call_{uuid.uuid4().hex[:8]}",
                        "name": "add",
                        "arguments": {"a": nums[0], "b": nums[1]},
                    }
                ]
            }
    if "time" in msg:
        return {
            "tool_calls": [
                {
                    "id": f"call_{uuid.uuid4().hex[:8]}",
                    "name": "get_time",
                    "arguments": {"timezone": "UTC"},
                }
            ]
        }
    match = re.search(r"weather in (\w+)", msg)
    if match:
        city = match.group(1).title()
        return {
            "tool_calls": [
                {
                    "id": f"call_{uuid.uuid4().hex[:8]}",
                    "name": "get_weather",
                    "arguments": {"city": city, "units": "celsius"},
                }
            ]
        }
    return {"content": "I cannot route that query to any registered tool."}


def run_loop(user_msg: str) -> None:
    print("=" * 72)
    print(f"USER : {user_msg}")
    print("-" * 72)
    tools_by_name = {t.name: t for t in REGISTRY}
    history: list[dict] = [{"role": "user", "content": user_msg}]
    for turn in range(1, MAX_TURNS + 1):
        decision = fake_decide(user_msg, history)
        if "content" in decision:
            print(f"TURN {turn} DECIDE : final answer")
            print(f"MODEL : {decision['content']}")
            return
        for call in decision["tool_calls"]:
            tool = tools_by_name.get(call["name"])
            print(f"TURN {turn} DECIDE : call {call['name']} id={call['id']}")
            print(f"           args = {json.dumps(call['arguments'])}")
            if tool is None:
                print(f"           ERROR : unknown tool {call['name']}")
                return
            errs = validate(tool.input_schema, call["arguments"])
            if errs:
                print(f"           VALIDATION ERRORS : {errs}")
                return
            if tool.consequential:
                print("           GATE : tool is consequential, would confirm")
            start = time.perf_counter()
            result = tool.executor(call["arguments"])
            ms = (time.perf_counter() - start) * 1000
            print(f"TURN {turn} EXECUTE: {tool.name} -> {json.dumps(result)}"
                  f" [{ms:.2f} ms]")
            history.append({
                "role": "tool", "id": call["id"],
                "name": tool.name, "content": json.dumps(result),
            })
        print(f"TURN {turn} OBSERVE: history length = {len(history)}")
    print("LOOP TERMINATED : hit MAX_TURNS circuit breaker")


def describe_registry() -> None:
    print("TOOL REGISTRY")
    print("-" * 72)
    for t in REGISTRY:
        kind = "consequential" if t.consequential else "pure"
        print(f"  {t.name:14s} [{kind}] - {t.description}")
    print()


def main() -> None:
    print("=" * 72)
    print("PHASE 13 LESSON 01 - THE TOOL INTERFACE")
    print("=" * 72)
    describe_registry()
    for query in (
        "please add 7 and 35",
        "what time is it?",
        "tell me the weather in Bengaluru",
        "write me a haiku about tea",
    ):
        run_loop(query)
        print()


if __name__ == "__main__":
    main()

PHASE 13 LESSON 01 - THE TOOL INTERFACE
TOOL REGISTRY
------------------------------------------------------------------------
  add            [pure] - Use when the user asks for the sum of two numbers. Do not use for subtraction, product, or symbolic algebra.
  get_time       [pure] - Use when the user asks what time it is. Do not use for historical dates or future scheduling.
  get_weather    [pure] - Use when the user asks about current conditions in a named city. Do not use for forecasts or historical weather data.

USER : please add 7 and 35
------------------------------------------------------------------------
TURN 1 DECIDE : call add id=call_cf118141
           args = {"a": 7.0, "b": 35.0}
TURN 1 EXECUTE: add -> {"sum": 42.0} [0.00 ms]
TURN 1 OBSERVE: history length = 2
TURN 2 DECIDE : final answer
MODEL : Final answer built from tool output: {"sum": 42.0}

USER : what time is it?
------------------------------------------------------------------------
TURN 1 DECIDE : call ge

In [5]:
import datetime as dt
import json
import re
import time
import uuid
from dataclasses import dataclass
from typing import Any, Callable

MAX_TURNS = 5

@dataclass
class Tool:
    name: str
    description: str
    input_schema: dict
    executor: Callable[[dict], Any]
    consequential: bool = False

# =====================================================================
# สุมฟังก์ชั่นรันงาน (Executors) ดั้งเดิมจากไฟล์ main.py
# =====================================================================
def tool_add(args: dict) -> dict:
    return {"sum": args["a"] + args["b"]}

def tool_get_time(args: dict) -> dict:
    tz = args.get("timezone", "UTC")
    now = dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds")
    return {"now": now, "timezone": tz}

def tool_get_weather(args: dict) -> dict:
    fake = {"Bengaluru": 28, "Tokyo": 12, "Zurich": 4, "Lagos": 31}
    city = args["city"]
    units = args.get("units", "celsius")
    temp = fake.get(city, 20)
    return {"city": city, "temp": temp, "units": units}

# 🛠️ [โจทย์ข้อ 1]: เขียนฟังก์ชั่นรันงานจริงสำหรับเครื่องมือดึงราคาหุ้นอันใหม่
def tool_get_stock_price(args: dict) -> dict:
    ticker = args["ticker"].upper()
    # จำลองการส่งค่าราคาหุ้นดิบกลับมาในรูปแบบ JSON
    fake_market = {"AAPL": 175.50, "GOOGL": 150.25, "MSFT": 420.10}
    price = fake_market.get(ticker, 100.00)
    return {"ticker": ticker, "current_price": price, "currency": "USD"}

# =====================================================================
# คลังลงทะเบียนเครื่องมือ (REGISTRY)
# =====================================================================
REGISTRY: list[Tool] = [
    Tool(
        name="add",
        description="Use when the user asks for the sum of two numbers. Do not use for subtraction, product, or symbolic algebra.",
        input_schema={"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"]},
        executor=tool_add,
    ),
    Tool(
        name="get_time",
        description="Use when the user asks what time it is. Do not use for historical dates or future scheduling.",
        input_schema={"type": "object", "properties": {"timezone": {"type": "string"}}, "required": []},
        executor=tool_get_time,
    ),
    Tool(
        name="get_weather",
        description="Use when the user asks about current conditions in a named city. Do not use for forecasts or historical weather data.",
        input_schema={"type": "object", "properties": {"city": {"type": "string"}, "units": {"type": "string", "enum": ["celsius", "fahrenheit"]}}, "required": ["city"]},
        executor=tool_get_weather,
    ),
    # 🛠️ [โจทย์ข้อ 1]: ลงทะเบียนเครื่องมืออันที่สี่ พร้อมคำอธิบายและ Schema ตรงตามเงื่อนไขใบงาน
    Tool(
        name="get_stock_price",
        description="Use when the user asks for a current stock price by ticker. Do not use for historical prices or market summaries.",
        input_schema={
            "type": "object",
            "properties": {
                "ticker": {"type": "string"}
            },
            "required": ["ticker"]
        },
        executor=tool_get_stock_price,
    )
]

# =====================================================================
# ตัวคัดกรองขยะจำลอง (Validator) อิงตามคลังข้อมูลหลักในโค้ด GitHub
# =====================================================================
def validate(schema: dict, value: Any) -> list[str]:
    errors: list[str] = []
    t = schema.get("type")
    if t == "object":
        if not isinstance(value, dict):
            return [f"expected object, got {type(value).__name__}"]
        for field in schema.get("required", []):
            if field not in value:
                errors.append(f"missing required field '{field}'")
        for key, sub in schema.get("properties", {}).items():
            if key in value:
                errors.extend(validate(sub, value[key]))
        return errors
    if t == "number" and not isinstance(value, (int, float)):
        errors.append(f"expected number, got {type(value).__name__}")
    if t == "string" and not isinstance(value, str):
        errors.append(f"expected string, got {type(value).__name__}")
    if "enum" in schema and value not in schema["enum"]:
        errors.append(f"value {value!r} not in enum {schema['enum']}")
    return errors

# =====================================================================
# ตัวตัดสินใจจำลอง (Fake Decider) สั่งจัดเส้นทางงานใหม่
# =====================================================================
import re
import uuid

def fake_decide(user_msg: str, history: list[dict]) -> dict:
    last = history[-1] if history else {}
    if last.get("role") == "tool":
        return {"content": f"Final answer built from tool output: {last.get('content')}"}

    msg = user_msg.lower()

    # 🎯 แก้ไขใหม่: ดักจับชื่อหุ้นยอดฮิตตรงๆ บ่ต้องใช้สุ่มเสี่ยงตัดคำมั่ว
    if any(keyword in msg for keyword in ["stock", "price", "หุ้น", "ราคา"]):
        ticker = "AAPL" # ค่าเริ่มต้นกันเหนียว

        if "googl" in msg or "โกเกิ้ล" in msg:
            ticker = "GOOGL"
        elif "msft" in msg or "ไมโครซอฟท์" in msg:
            ticker = "MSFT"
        elif "aapl" in msg or "แอปเปิ้ล" in msg:
            ticker = "AAPL"

        return {
            "tool_calls": [
                {
                    "id": f"call_{uuid.uuid4().hex[:8]}",
                    "name": "get_stock_price",
                    "arguments": {"ticker": ticker}
                }
            ]
        }

    # โค้ดส่วนอื่นๆ (add, get_time, get_weather) ปล่อยไว้คงเดิม...

    # สุมระบบดักจับแบบเดิมในคลังไฟล์ main.py
    if re.search(r"\b(add|sum|plus)\b", msg):
        nums = [float(n) for n in re.findall(r"-?\d+\.?\d*", msg)]
        if len(nums) >= 2:
            return {"tool_calls": [{"id": f"call_{uuid.uuid4().hex[:8]}", "name": "add", "arguments": {"a": nums, "b": nums}}]}
    if "time" in msg:
        return {"tool_calls": [{"id": f"call_{uuid.uuid4().hex[:8]}", "name": "get_time", "arguments": {"timezone": "UTC"}}]}
    match = re.search(r"weather in (\w+)", msg)
    if match:
        city = match.group(1).title()
        return {"tool_calls": [{"id": f"call_{uuid.uuid4().hex[:8]}", "name": "get_weather", "arguments": {"city": city, "units": "celsius"}}]}

    return {"content": "I cannot route that query to any registered tool."}

# =====================================================================
# ตัวคุมระบบวนลูป Host (The Core Engine)
# =====================================================================
def run_loop(user_msg: str) -> None:
    print("=" * 72)
    print(f"USER : {user_msg}")
    print("-" * 72)
    tools_by_name = {t.name: t for t in REGISTRY}
    history: list[dict] = [{"role": "user", "content": user_msg}]

    for turn in range(1, MAX_TURNS + 1):
        decision = fake_decide(user_msg, history)
        if "content" in decision:
            print(f"TURN {turn} DECIDE : final answer")
            print(f"MODEL : {decision['content']}")
            return

        for call in decision["tool_calls"]:
            tool = tools_by_name.get(call["name"])
            print(f"TURN {turn} DECIDE : call {call['name']} id={call['id']}")
            print(f"           args = {json.dumps(call['arguments'])}")
            if tool is None:
                print(f"           ERROR : unknown tool {call['name']}")
                return

            errs = validate(tool.input_schema, call["arguments"])
            if errs:
                print(f"           VALIDATION ERRORS : {errs}")
                return

            start = time.perf_counter()
            result = tool.executor(call["arguments"])
            ms = (time.perf_counter() - start) * 1000
            print(f"TURN {turn} EXECUTE: {tool.name} -> {json.dumps(result)} [{ms:.2f} ms]")

            history.append({"role": "tool", "id": call["id"], "name": tool.name, "content": json.dumps(result)})
            print(f"TURN {turn} OBSERVE: history length = {len(history)}")

    print("LOOP TERMINATED : hit MAX_TURNS circuit breaker")

def describe_registry() -> None:
    print("TOOL REGISTRY")
    print("-" * 72)
    for t in REGISTRY:
        kind = "consequential" if t.consequential else "pure"
        print(f"  {t.name:14s} [{kind}] - {t.description}")
    print()

# =====================================================================
# รันเช็คผลลัพธ์หน้างานบนหน้าจอ Colab
# =====================================================================
describe_registry()

# 🧪 ทดสอบยิงคำสั่งถามเรื่องหุ้น เพื่อยืนยันว่าระบบจัดเส้นทาง (Fake Decider Routes) ส่งงานเข้าเครื่องมืออันใหม่จริงบ่
run_loop("Please check the current stock price of AAPL")
run_loop("อยากฮู้ราคา หุ้น โกเกิ้ล (GOOGL) ตอนนี้แหมะ")

TOOL REGISTRY
------------------------------------------------------------------------
  add            [pure] - Use when the user asks for the sum of two numbers. Do not use for subtraction, product, or symbolic algebra.
  get_time       [pure] - Use when the user asks what time it is. Do not use for historical dates or future scheduling.
  get_weather    [pure] - Use when the user asks about current conditions in a named city. Do not use for forecasts or historical weather data.
  get_stock_price [pure] - Use when the user asks for a current stock price by ticker. Do not use for historical prices or market summaries.

USER : Please check the current stock price of AAPL
------------------------------------------------------------------------
TURN 1 DECIDE : call get_stock_price id=call_81e5fd43
           args = {"ticker": "AAPL"}
TURN 1 EXECUTE: get_stock_price -> {"ticker": "AAPL", "current_price": 175.5, "currency": "USD"} [0.00 ms]
TURN 1 OBSERVE: history length = 2
TURN 2 DECIDE 

In [6]:
import json

# ดึงฟังก์ชั่น validate ต้นฉบับมาจากคลังสคริปต์ใน GitHub ของหน้าเว็บ
def validate(schema: dict, value: Any) -> list[str]:
    errors: list[str] = []
    t = schema.get("type")
    if t == "object":
        if not isinstance(value, dict):
            return [f"expected object, got {type(value).__name__}"]

        # 1. เช็คฟิลด์บังคับ (Required Fields)
        for field in schema.get("required", []):
            if field not in value:
                errors.append(f"missing required field '{field}'")

        # 2. เช็คฟิลด์แปลกปลอม (Extra/Unknown Fields)
        # ขยายความสามารถให้ตัว Validator ดักจับฟิลด์ที่ไม่ได้ประกาศไว้ใน properties ทันที
        properties = schema.get("properties", {})
        for key in value.keys():
            if key not in properties:
                errors.append(f"❌ [Security Violations]: พบฟิลด์แปลกปลอม '{key}' ที่ไม่ได้ลงทะเบียนไว้!")

        for key, sub in properties.items():
            if key in value:
                errors.extend(validate(sub, value[key]))
        return errors
    return errors

# โครงสร้าง Schema ของเครื่องมือ get_stock_price ที่เฮาทำไว้ในโจทย์ข้อ 1
STOCK_SCHEMA = {
    "type": "object",
    "properties": {
        "ticker": {"type": "string"}
    },
    "required": ["ticker"]
}

# ---------------------------------------------------------------------
# 📊 ด่านทดสอบรันหน้างานบน Colab
# ---------------------------------------------------------------------

print("=== 🛑 เคสที่ 1: แกล้งส่งอาร์กิวเมนต์แบบ ขาดฟิลด์บังคับ (Missing Required Field) ===")
broken_args_1 = {}  # ว่างเปล่า บ่ยอมส่ง 'ticker' มาให้
errs_1 = validate(STOCK_SCHEMA, broken_args_1)
if errs_1:
    print(f"🛡️ [Host REJECTED]: ระบบทำการตีตกก่อนรันงานจริง! บั๊กที่พบ: {errs_1}")
else:
    print("🔓 [⚠️ เจ๊ง]: ปล่อยให้ผ่านไปได้จั่งใด๋วะ!")

print("\n" + "="*72 + "\n")

print("=== 🛑 เคสที่ 2: แกล้งส่งฟิลด์แปลกปลอมยัดไส้เข้ามา (Extra Unknown Field) ===")
# ส่ง ticker มาครบก็จริง แต่วิศวกรสายมืดแอบยัดไส้ฟิลด์แฮกระบบแฝงตัวมานำ
broken_args_2 = {"ticker": "AAPL", "malicious_payload": "DROP TABLE Users;"}
errs_2 = validate(STOCK_SCHEMA, broken_args_2)
if errs_2:
    print(f"🛡️ [Host REJECTED]: ระบบทำการตีตกเรียบร้อย! บั๊กที่พบ: {errs_2}")
else:
    print("🔓 [⚠️ เจ๊ง]: ปล่อยเบลอให้ฟิลด์แปลกปลอมหลุดเข้าไปหา Executor!")

=== 🛑 เคสที่ 1: แกล้งส่งอาร์กิวเมนต์แบบ ขาดฟิลด์บังคับ (Missing Required Field) ===
🛡️ [Host REJECTED]: ระบบทำการตีตกก่อนรันงานจริง! บั๊กที่พบ: ["missing required field 'ticker'"]


=== 🛑 เคสที่ 2: แกล้งส่งฟิลด์แปลกปลอมยัดไส้เข้ามา (Extra Unknown Field) ===
🛡️ [Host REJECTED]: ระบบทำการตีตกเรียบร้อย! บั๊กที่พบ: ["❌ [Security Violations]: พบฟิลด์แปลกปลอม 'malicious_payload' ที่ไม่ได้ลงทะเบียนไว้!"]


In [7]:
import json
import time

# =====================================================================
# 1. ปรับโครงสร้างคลาส Tool ให้รองรับแฟล็กพ่วงท้าย (ตามโจทย์สั่ง)
# =====================================================================
class Tool:
    def __init__(self, name: str, description: str, input_schema: dict, executor: callable, consequential: bool = False):
        self.name = name
        self.description = description
        self.input_schema = input_schema
        self.executor = executor
        self.consequential = consequential  # 🚩 แฟล็กแยกประเภท: True = สายลุย / False = สายสะอาด

# =====================================================================
# 2. ฟังก์ชั่นจำลองระบบรันงาน (Executors)
# =====================================================================
def tool_add(args): return {"sum": args["a"] + args["b"]}
def tool_get_time(args): return {"now": "2026-06-06T19:33:00Z"}
def tool_get_weather(args): return {"city": args["city"], "temp": 28}
def tool_get_stock_price(args): return {"ticker": args["ticker"], "price": 175.50}

# 🔥 เครื่องมืออันตรายตัวใหม่: จำลองการลบไฟล์ระบบ หรือตัดเงินบัญชี
def tool_delete_file(args): return {"status": "success", "deleted_file": args["filename"]}

# =====================================================================
# 3. [โจทย์พาร์ท 1 & 2]: จัดกลุ่มเครื่องมือ (Classify) และยัดธงใส่ออพชั่น
# =====================================================================
REGISTRY = {
    "add": Tool(
        name="add",
        description="บวกเลขธรรมดา",
        input_schema={"type": "object", "properties": {"a": {"type": "number"}, "b": {"type": "number"}}, "required": ["a", "b"]},
        executor=tool_add,
        consequential=False  # ✅ [Pure] อ่าน/คำนวณในร่ม ปลอดภัยหายห่วง
    ),
    "get_time": Tool(
        name="get_time",
        description="ดูเวลาปัจจุบัน",
        input_schema={"type": "object", "properties": {}},
        executor=tool_get_time,
        consequential=False  # ✅ [Pure] ดึงเวลามาดูเฉยๆ บ่พังหยังดอก
    ),
    "get_weather": Tool(
        name="get_weather",
        description="ดูสภาพอากาศ",
        input_schema={"type": "object", "properties": {"city": {"type": "string"}}, "required": ["city"]},
        executor=tool_get_weather,
        consequential=False  # ✅ [Pure] แอบซอมสภาพอากาศเฉยๆ บ่มีผลกระทบ
    ),
    "get_stock_price": Tool(
        name="get_stock_price",
        description="ดูราคาหุ้นล่าสุด",
        input_schema={"type": "object", "properties": {"ticker": {"type": "string"}}, "required": ["ticker"]},
        executor=tool_get_stock_price,
        consequential=False  # ✅ [Pure] ดึงราคามาส่องเบิ่งเฉยๆ บ่ใช่การกดส่งคำสั่งซื้อขาย
    ),
    # ⚠️ เพิ่มเครื่องมือสายลุย (Consequential) เข้ามาทดสอบระบบประตูกั้นตัวใหม่
    "delete_file": Tool(
        name="delete_file",
        description="ลบไฟล์งานออกจากเครื่องคอมพิวเตอร์",
        input_schema={"type": "object", "properties": {"filename": {"type": "string"}}, "required": ["filename"]},
        executor=tool_delete_file,
        consequential=True   # 🛑 [Consequential: true] อันตรายคัก! ต้องตั้งการ์ดกั้นไว้!
    )
}

# =====================================================================
# 4. [โจทย์พาร์ท 3]: ปรับแต่งลูปควบคุมหลัก (Host Loop) ให้มีด่านตรวจประตูกั้น
# =====================================================================
def run_host_loop_with_gate(tool_name: str, args: dict):
    print(f"🧠 [โมเดลสั่ง]: เอิ้นใช้เครื่องมือ [{tool_name}] พร้อมอาร์กิวเมนต์ {json.dumps(args)}")

    # วิ่งไปควานหาเครื่องมือในคลังแสง
    tool = REGISTRY.get(tool_name)
    if not tool:
        print("❌ บ่พบเครื่องมือนี้ในระบบ")
        return

    # 🛡️ [Confirmation Gate]: คั่นเจอสายลุยปั๊บ สั่งเบรกและพ่นคำสั่งเตือนทันที!
    if tool.consequential:
        print(f"⚠️  [ประตูกั้นทำงาน]: พบเครื่องมือที่มีผลกระทบข้างเคียงสูง!")
        print(f"    👉 สั่งพ่นโค้ดระวังภัยล่วงหน้า -> \"would confirm with user\"")
        print(f"    📢 [Host Status]: ระบบจะหยุดค้างรอให้มนุษย์มากดยืนยัน (Confirm) หรือยกเลิก (Cancel) ก่อนลงมือรันจริง!")
        # ในแอปจริงตรงนี้สิเด้ง Pop-up หน้าจอให้คนกดปุ่ม คั่นมนุษย์เซย์เยส จั่งสิปล่อยให้วิ่งไปหา Executor
    else:
        print(f"🟢 [ประตูกั้นปล่อยผ่าน]: เครื่องมือ [{tool_name}] เป็นสายสะอาด (Pure) รันงานออโต้ได้ทันที")

    # ด่านลงมือทำจริง (Execute)
    start_time = time.perf_counter()
    result = tool.executor(args)
    ms = (time.perf_counter() - start_time) * 1000
    print(f"👁️  [ผลลัพธ์จากเครื่องมือ]: {json.dumps(result)} [{ms:.2f} ms]")
    print("-" * 72)

# ---------------------------------------------------------------------
# 📊 รันทำการทดสอบหน้าจอ Colab เพื่อดูความแตกต่าง
# ---------------------------------------------------------------------
print("=== 🧪 เคสทดสอบที่ 1: เรียกใช้เครื่องมือสายสะอาด (Pure Tool) ===")
run_host_loop_with_gate("get_stock_price", {"ticker": "AAPL"})

print("=== 🧪 เคสทดสอบที่ 2: เรียกใช้เครื่องมือสายลุยอันตราย (Consequential Tool) ===")
run_host_loop_with_gate("delete_file", {"filename": "database_backup.sql"})

=== 🧪 เคสทดสอบที่ 1: เรียกใช้เครื่องมือสายสะอาด (Pure Tool) ===
🧠 [โมเดลสั่ง]: เอิ้นใช้เครื่องมือ [get_stock_price] พร้อมอาร์กิวเมนต์ {"ticker": "AAPL"}
🟢 [ประตูกั้นปล่อยผ่าน]: เครื่องมือ [get_stock_price] เป็นสายสะอาด (Pure) รันงานออโต้ได้ทันที
👁️  [ผลลัพธ์จากเครื่องมือ]: {"ticker": "AAPL", "price": 175.5} [0.00 ms]
------------------------------------------------------------------------
=== 🧪 เคสทดสอบที่ 2: เรียกใช้เครื่องมือสายลุยอันตราย (Consequential Tool) ===
🧠 [โมเดลสั่ง]: เอิ้นใช้เครื่องมือ [delete_file] พร้อมอาร์กิวเมนต์ {"filename": "database_backup.sql"}
⚠️  [ประตูกั้นทำงาน]: พบเครื่องมือที่มีผลกระทบข้างเคียงสูง!
    👉 สั่งพ่นโค้ดระวังภัยล่วงหน้า -> "would confirm with user"
    📢 [Host Status]: ระบบจะหยุดค้างรอให้มนุษย์มากดยืนยัน (Confirm) หรือยกเลิก (Cancel) ก่อนลงมือรันจริง!
👁️  [ผลลัพธ์จากเครื่องมือ]: {"status": "success", "deleted_file": "database_backup.sql"} [0.00 ms]
------------------------------------------------------------------------
